In [1]:
import polars as pl
from glob import glob

In [2]:
def show_all(df, width=200):
    '''
    Prints an entire polars dataframe in the console or notebook output.
    
    Parameters
    ----------
    df : pl.DataFrame
        The dataframe to be printed.
    width : int, optional
        The width (in characters) of the printed dataframe.
        Defaults to 200.
    '''
    with pl.Config() as cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        cfg.set_fmt_str_lengths(width)
        
        print(df)

In [3]:
globdb_tax = pl.read_csv('/mnt/hpccs01/work/microbiome/db/globdb/232/globdb_r232_taxonomy.tsv', separator='\t', new_columns=['acc', 'tax'])
globdb_score = (
    pl.read_csv('/mnt/hpccs01/work/microbiome/db/globdb/232/globdb_r232_checkm2.tsv', separator='\t')
    .select(
        pl.col('Name').alias('acc'), 
        pl.col('Completeness').alias('score')
    )
    )

In [4]:
asgards = (
    globdb_tax
    .filter(pl.col("tax").str.contains("p__Asgardarchaeota"))
    # .head()
    .with_columns(pl.col("tax").str.split(";", inclusive=False).list.get(2).alias("class"))
    .join(globdb_score, on="acc", how="left")
    .sort("score", descending=True)
    .unique(subset=["class"], keep="first")
    .with_columns(
        pl.col('acc').map_elements(lambda acc: glob(f'/work/microbiome/db/globdb/232/globdb_r232_genome_fasta/*/{acc}.fa.gz')[0], return_dtype=pl.Utf8).alias('path')
    )
    )

print(asgards)

asgards.write_csv('metadata/asgard_genomes.csv')

shape: (13, 5)
┌─────────────────┬────────────────────────┬──────────────────────┬───────┬────────────────────────┐
│ acc             ┆ tax                    ┆ class                ┆ score ┆ path                   │
│ ---             ┆ ---                    ┆ ---                  ┆ ---   ┆ ---                    │
│ str             ┆ str                    ┆ str                  ┆ f64   ┆ str                    │
╞═════════════════╪════════════════════════╪══════════════════════╪═══════╪════════════════════════╡
│ GCA_040225955   ┆ d__Archaea;p__Asgardar ┆ c__Asgardarchaeia    ┆ 93.49 ┆ /work/microbiome/db/gl │
│                 ┆ chaeota;…              ┆                      ┆       ┆ obdb/232…              │
│ GCA_030587605   ┆ d__Archaea;p__Asgardar ┆ c__Njordarchaeia     ┆ 90.51 ┆ /work/microbiome/db/gl │
│                 ┆ chaeota;…              ┆                      ┆       ┆ obdb/232…              │
│ GCMETA_00802690 ┆ d__Archaea;p__Asgardar ┆ c__Baldrarchaeia     ┆ 97.13 ┆ 

In [5]:
OUTGROUP_GENOME_MAP = dict(
    asgards.select(
            pl.col("acc").cast(pl.Utf8),
            pl.col("path").cast(pl.Utf8),
        )
        .iter_rows()
    )

In [10]:
import dendropy

# Load the tree with force-rooted option to properly handle tree state
tree = dendropy.Tree.get(
    path="output/tree/tree.nwk", 
    schema="newick", 
    preserve_underscores=True, 
    rooting="force-rooted"
)

print(tree.taxon_namespace)

# Filter keys from the global OUTGROUP_GENOME_MAP that exist in the tree
valid_taxa = [acc for acc in OUTGROUP_GENOME_MAP.keys() if acc in tree.taxon_namespace.labels()]

print(valid_taxa)


if valid_taxa:
    # Find the Most Recent Common Ancestor (MRCA) of these outgroup genomes
    mrca_node = tree.mrca(taxon_labels=valid_taxa)
    
    if mrca_node and mrca_node.edge:
        edge_length = mrca_node.edge.length
        length_split = 0.5 * edge_length if edge_length is not None else None
        
        # Reroot the tree at the edge leading to the MRCA clade
        tree.reroot_at_edge(
            mrca_node.edge, 
            length1=length_split, 
            length2=length_split, 
            update_bipartitions=False
        )

['Tricam1', 'Trigui1', 'TriharM10_1', 'Trichpleur1', 'Tricom1', 'Tri5640_1', 'Tripare1', 'Triag1', 'Trialn1', 'Triheli1', 'Trispi1_2', 'Trichl1', 'Tricre1', 'TriviGv29_8_2', 'Tricer1', 'Trist1', 'Triflo1', 'Tricae1', 'Triros1', 'Tribar1', 'Trifer1', 'Triaru1', 'Tribrev1', 'Hypper1', 'Mycosp1', 'Parmar1', 'Valla1', 'Sarkil1', 'Sarstr1', 'Neodi1', 'Ilyeu1', 'Neora1', 'Daces1', 'Dacma1', 'Cylol1', 'MarPMI226', 'Fusdim1', 'Fusavent1', 'Neobo1', 'Fuscuc1', 'Fusso1', 'FusspF23_1', 'Necha3', 'FvanT343_1', 'Fus66182_1', 'Fuslat1', 'Fuscon1', 'Fusre1', 'Fusnis1', 'Fusnyg1', 'Fusve2', 'Fusprol1', 'Fussub1', 'FusoxFo47_2', 'Fusco1', 'Fusacu1', 'Fusave1', 'Fussul1', 'Fuseq1', 'Fusequ1', 'Fuspoa1', 'Fusven1', 'Fusspo1', 'Fusgr2_1', 'Fuspse1', 'Nieex1', 'Stael1', 'Stach1', 'Stachl1', 'Myrin1', 'binchicken_co10078.50_euk', 'Melti1', 'Torra1', 'Brefa1', 'Thipu1', 'Corma2', 'binchicken_co6527.211_euk', 'Mictr1', 'Theste1', 'Lompr1', 'Sceap1', 'binchicken_co4874.175_euk', 'binchicken_co5754.33_euk', 'Co

In [11]:
print(OUTGROUP_GENOME_MAP)

{'GCA_040225955': '/work/microbiome/db/globdb/232/globdb_r232_genome_fasta/R232CHUNK008/GCA_040225955.fa.gz', 'GCA_030587605': '/work/microbiome/db/globdb/232/globdb_r232_genome_fasta/R232CHUNK006/GCA_030587605.fa.gz', 'GCMETA_00802690': '/work/microbiome/db/globdb/232/globdb_r232_genome_fasta/R232CHUNK022/GCMETA_00802690.fa.gz', 'GCA_048937405': '/work/microbiome/db/globdb/232/globdb_r232_genome_fasta/R232CHUNK010/GCA_048937405.fa.gz', 'GCA_016840425': '/work/microbiome/db/globdb/232/globdb_r232_genome_fasta/R232CHUNK003/GCA_016840425.fa.gz', 'GCA_019057385': '/work/microbiome/db/globdb/232/globdb_r232_genome_fasta/R232CHUNK003/GCA_019057385.fa.gz', 'GCA_048938295': '/work/microbiome/db/globdb/232/globdb_r232_genome_fasta/R232CHUNK010/GCA_048938295.fa.gz', 'GCA_037310415': '/work/microbiome/db/globdb/232/globdb_r232_genome_fasta/R232CHUNK007/GCA_037310415.fa.gz', 'GCA_037305845': '/work/microbiome/db/globdb/232/globdb_r232_genome_fasta/R232CHUNK007/GCA_037305845.fa.gz', 'MOTU40_067193

In [ ]:
# metadata = (
#     pl.read_csv(input.tree
#         "/mnt/hpccs01/home/n9985158/trees/taxonomy.tsv", separator="\t", 
#         has_header=False, new_columns=['genome_id', 'taxonomy'])
#     .with_columns(
#         pl.lit('known species').alias('novelty'))
#         )

metadata = (
    pl.read_csv(
        "/mnt/hpccs01/home/n9985158/trees/taxonomy.revised_reference.tsv", separator="\t", 
        # has_header=False, new_columns=['genome_id', 'taxonomy']
        )
    .with_columns(
        pl.lit('known species').alias('novelty'))
        )

new_tax = (
    pl.read_csv("/mnt/hpccs01/home/n9985158/trees/target_novelty_taxonomy.tsv", separator="\t")
    .select(['genome_id', 'taxonomy', 'novelty'])
    )

print(metadata.head())
print(new_tax.head())

shape: (5, 3)
┌─────────────────┬─────────────────────────────────┬───────────────┐
│ genome_id       ┆ taxonomy                        ┆ novelty       │
│ ---             ┆ ---                             ┆ ---           │
│ str             ┆ str                             ┆ str           │
╞═════════════════╪═════════════════════════════════╪═══════════════╡
│ GCA_018295905.1 ┆ d__Eukarya;p__Ciliophora_A;c__… ┆ known species │
│ GCA_946863485.1 ┆ d__Eukarya;p__Ciliophora_A;c__… ┆ known species │
│ GCA_001880345.2 ┆ d__Eukarya;p__Ciliophora_A;c__… ┆ known species │
│ GCA_048569255.1 ┆ d__Eukarya;p__Ciliophora_A;c__… ┆ known species │
│ GCA_027382585.1 ┆ d__Eukarya;p__Ciliophora_A;c__… ┆ known species │
└─────────────────┴─────────────────────────────────┴───────────────┘
shape: (5, 3)
┌─────────────────┬─────────────────────────────────┬─────────────┐
│ genome_id       ┆ taxonomy                        ┆ novelty     │
│ ---             ┆ ---                             ┆ ---         

In [65]:
print(
    pl.read_csv('/mnt/hpccs01/home/n9985158/trees/tree_metadata.csv')
)

shape: (7_229, 25)
┌────────────┬─────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ acc        ┆ ncbi_id ┆ parsed_ta ┆ taxonomy_ ┆ … ┆ mag_sourc ┆ eukcc_tax ┆ eukcc_dir ┆ taxonomy_ │
│ ---        ┆ ---     ┆ xon_strin ┆ source    ┆   ┆ e_colour  ┆ onomy     ┆ ---       ┆ match_sta │
│ str        ┆ i64     ┆ g         ┆ ---       ┆   ┆ ---       ┆ ---       ┆ str       ┆ tus       │
│            ┆         ┆ ---       ┆ str       ┆   ┆ str       ┆ str       ┆           ┆ ---       │
│            ┆         ┆ str       ┆           ┆   ┆           ┆           ┆           ┆ str       │
╞════════════╪═════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Dipaus_NRR ┆ 43976   ┆ d__;      ┆ ncbi_requ ┆ … ┆ #808080   ┆ NA        ┆ NA        ┆ not_avail │
│ LY17565_1  ┆         ┆ k__Fungi; ┆ ery       ┆   ┆           ┆           ┆           ┆ able      │
│            ┆         ┆ p__Ascomy ┆           ┆   ┆           ┆        

In [66]:
show_all(new_tax['taxonomy'])

shape: (578,)
Series: 'taxonomy' [str]
[
	"d__Eukarya;p__Ciliophora_A;c__Spirotrichea_C;o__Spirotrichea_C_o1;f__Spirotrichea_C_o1_f1;g__Spirotrichea_C_o1_f1_g1;s__Spirotrichea_C sp_1"
	"d__Eukarya;p__Ciliophora_A;c__Heterotrichea;o__Heterotrichida_B;f__Stentoridae_B;g__Stentoridae_B_g1;s__Stentoridae_B sp_1"
	"d__Eukarya;p__Ciliophora_B;c__Ciliophora_B_c1;o__Ciliophora_B_c1_o1;f__Ciliophora_B_c1_o1_f1;g__Ciliophora_B_c1_o1_f1_g1;s__Ciliophora_B sp_1"
	"d__Eukarya;p__Ciliophora_B;c__Ciliophora_B_c2;o__Ciliophora_B_c2_o1;f__Ciliophora_B_c2_o1_f1;g__Ciliophora_B_c2_o1_f1_g1;s__Ciliophora_B sp_1"
	"d__Eukarya;p__Ciliophora_B;c__Ciliophora_B_c3;o__Ciliophora_B_c3_o1;f__Ciliophora_B_c3_o1_f1;g__Ciliophora_B_c3_o1_f1_g1;s__Ciliophora_B sp_1"
	"d__Eukarya;p__Eukarya_p1;c__Eukarya_p1_c1;o__Eukarya_p1_c1_o1;f__Eukarya_p1_c1_o1_f1;g__Eukarya_p1_c1_o1_f1_g1;s__Eukarya sp_1"
	"d__Eukarya;p__Eukarya_p2;c__Eukarya_p2_c1;o__Eukarya_p2_c1_o1;f__Eukarya_p2_c1_o1_f1;g__Eukarya_p2_c1_o1_f1_g1;s__Eukarya s

In [67]:
import random

full_metadata = (
    pl.concat([
    metadata
        .with_columns(
            pl.col('taxonomy').str.split(';').list.get(0).alias('domain'),
            pl.col('taxonomy').str.split(';').list.get(1).alias('phylum'),
            pl.col('taxonomy').str.split(';').list.get(2).alias('class'),
            pl.col('taxonomy').str.split(';').list.get(3).alias('order'),
            pl.col('taxonomy').str.split(';').list.get(4).alias('family'),
            pl.col('taxonomy').str.split(';').list.get(5).alias('genus'),
            pl.col('taxonomy').str.split(';').list.get(6).alias('species')),
    new_tax
        .with_columns(
            pl.col('taxonomy').str.split(';').list.get(0).alias('domain'),
            pl.col('taxonomy').str.split(';').list.get(1).alias('phylum'),
            pl.col('taxonomy').str.split(';').list.get(2).alias('class'),
            pl.col('taxonomy').str.split(';').list.get(3).alias('order'),
            pl.col('taxonomy').str.split(';').list.get(4).alias('family'),
            pl.col('taxonomy').str.split(';').list.get(5).alias('genus'),
            pl.col('taxonomy').str.split(';').list.get(6).alias('species')
            )
    ]
    )
    .unique(subset=['genome_id'])
)

novelty_colour_map = (
    full_metadata
    .select('novelty')
    .unique()
    .with_columns(
        pl.col('novelty')
        .map_elements(
            lambda _: f"#{random.randint(0, 0xFFFFFF):06x}",
            return_dtype=pl.Utf8,
        )
        .alias(f"colour")
        )
)

full_metadata = (
    full_metadata
    .join(novelty_colour_map, on='novelty', how='left')
)

print(full_metadata.unique('novelty'))

full_metadata.write_csv('/mnt/hpccs01/home/n9985158/trees/novelty_colours.csv')

shape: (7, 11)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬─────────┐
│ genome_id  ┆ taxonomy  ┆ novelty   ┆ domain    ┆ … ┆ family    ┆ genus     ┆ species   ┆ colour  │
│ ---        ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---     │
│ str        ┆ str       ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str     │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═════════╡
│ binchicken ┆ d__Eukary ┆ novel     ┆ d__Eukary ┆ … ┆ f__Lecano ┆ g__Lecano ┆ s__Lecano ┆ #4de7bf │
│ _co5849.10 ┆ a;p__Asco ┆ genus     ┆ a         ┆   ┆ rales_f2  ┆ rales_f2_ ┆ rales     ┆         │
│ 9_euk      ┆ mycota_D; ┆           ┆           ┆   ┆           ┆ g1        ┆ sp_1      ┆         │
│            ┆ c__…      ┆           ┆           ┆   ┆           ┆           ┆           ┆         │
│ binchicken ┆ d__Eukary ┆ novel     ┆ d__Eukary ┆ … ┆ f__Pichia ┆ g__Pichia